In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# 1. Dataset Simulation (Simplified Waterbirds for demonstration)
# In Waterbirds, most waterbirds are on water and landbirds on land.
# A small fraction (minority) are landbirds on water or waterbirds on land.
def create_waterbirds_batch(batch_size=256, minority_ratio=0.05):
    # Features: [bird_type, background_type, noise...]
    # y=0: Landbird, y=1: Waterbird
    # a=0: Land, a=1: Water
    batch_y = torch.randint(0, 2, (batch_size,))
    batch_a = torch.zeros(batch_size)
    
    for i in range(batch_size):
        if torch.rand(1) < (1 - minority_ratio):
            batch_a[i] = batch_y[i] # Majority: background matches bird
        else:
            batch_a[i] = 1 - batch_y[i] # Minority: background mismatches
    
    # Simple feature vector: bird feature + background feature + random noise
    x = torch.randn(batch_size, 10)
    x[:, 0] = batch_y.float() * 2.0  # Core feature (Stable)
    x[:, 1] = batch_a.float() * 5.0  # Spurious feature (Stronger gradient)
    return x, batch_y.long(), batch_a.long()


class LogitCorrectionLoss(nn.Module):
    def __init__(self, num_classes=2, momentum=0.5):
        super(LogitCorrectionLoss, self).__init__()
        self.num_classes = num_classes
        # The paper uses moving average for group prior estimation [cite: 202, 506]
        self.register_buffer('group_prior', torch.ones(num_classes, num_classes) / (num_classes**2))
        self.alpha = momentum # Momentum alpha set to 0.5 in paper [cite: 506]

    def forward(self, logits, labels, erm_logits):
        # The ERM network prediction estimates the spurious attribute P(a|x) [cite: 210, 215]
        erm_probs = F.softmax(erm_logits, dim=1)
        # a_x is estimated as the argmax of the ERM model's probability outputs [cite: 215, 498]
        attr_est = torch.argmax(erm_probs, dim=1)
        
        # Update group priors using the moving average strategy [cite: 202, 498]
        with torch.no_grad():
            # Get current batch estimation for P(y, a) [cite: 195, 202]
            batch_prior = torch.zeros_like(self.group_prior)
            for y in range(self.num_classes):
                for a in range(self.num_classes):
                    mask = (labels == y) & (attr_est == a)
                    if mask.any():
                        # Update based on ERM probability outputs as shown in Algorithm 1 [cite: 498]
                        batch_prior[y, a] = erm_probs[mask, a].mean()
            
            # Moving average update: P = alpha*P + (1-alpha)*batch_P [cite: 498, 506]
            self.group_prior = self.alpha * self.group_prior + (1 - self.alpha) * batch_prior

        # Logit Correction: L_LC = log(sum(exp(f + ln P))) - (f_y + ln P_y) [cite: 145]
        # This is equivalent to applying Cross Entropy on shifted logits [cite: 142, 143]
        eps = 1e-9
        # The correction term is ln(P_y,ax) [cite: 140, 142]
        correction = torch.log(self.group_prior + eps)
        
        # Correct each logit for the robust network [cite: 142, 498]
        # logits: (batch, L), correction: (L, K). We need the column for estimated a_x.
        corrected_logits = logits + correction[:, attr_est].T 
        
        return F.cross_entropy(corrected_logits, labels)

# Implementation of Generalized Cross Entropy (GCE) for the ERM branch [cite: 94, 123]
def gce_loss(logits, labels, q=0.7):
    probs = F.softmax(logits, dim=1)
    p_y = probs[torch.arange(labels.size(0)), labels]
    # Equation 3: (1 - p^q) / q [cite: 94]
    return torch.mean((1 - p_y**q) / q)

# 3. Model Definition (ResNet-50 is standard for Waterbirds) [cite: 7]
class TwoBranchModel(nn.Module):
    def __init__(self, input_dim=10, num_classes=2):
        super(TwoBranchModel, self).__init__()
        self.erm_branch = nn.Linear(input_dim, num_classes) # Biased Branch
        self.robust_branch = nn.Linear(input_dim, num_classes) # Robust Branch

    def forward(self, x):
        return self.erm_branch(x), self.robust_branch(x)

# 4. Training Loop
def train_model(use_lc=False):
    model = TwoBranchModel()
    optimizer = optim.Adam(model.parameters(), lr=1e-3) # [cite: 7]
    lc_criterion = LogitCorrectionLoss()
    
    for epoch in range(50):
        x, y, a = create_waterbirds_batch()
        optimizer.zero_grad()
        
        erm_logits, robust_logits = model(x)
        
        # ERM Branch always uses GCE or CE 
        # Generalized Cross Entropy (GCE) emphasizes majority groups [cite: 3]
        q = 0.8 # Hyperparameter for Waterbirds [cite: 7]
        erm_probs = F.softmax(erm_logits, dim=1)
        p_y = erm_probs[torch.arange(y.size(0)), y]
        loss_erm = torch.mean((1 - p_y**q) / q)
        
        if use_lc:
            # Logit Correction uses ERM outputs to debias [cite: 4]
            loss_robust = lc_criterion(robust_logits, y, erm_probs)
        else:
            # Standard ERM baseline uses standard Cross Entropy
            loss_robust = F.cross_entropy(robust_logits, y)
            
        total_loss = loss_erm + loss_robust
        total_loss.backward()
        optimizer.step()
        
    return model

# Example Usage
print("Training Baseline ERM...")
model_erm = train_model(use_lc=False)

print("Training with Logit Correction (LC)...")
model_lc = train_model(use_lc=True)

Training Baseline ERM...
Training with Logit Correction (LC)...


In [4]:
print(model_erm)
print(model_lc)

TwoBranchModel(
  (erm_branch): Linear(in_features=10, out_features=2, bias=True)
  (robust_branch): Linear(in_features=10, out_features=2, bias=True)
)
TwoBranchModel(
  (erm_branch): Linear(in_features=10, out_features=2, bias=True)
  (robust_branch): Linear(in_features=10, out_features=2, bias=True)
)


In [5]:
def evaluate_robustness(model, num_samples=1000):
    model.eval()
    # Waterbirds has 4 groups: (y=0, a=0), (y=0, a=1), (y=1, a=0), (y=1, a=1)
    group_counts = torch.zeros(2, 2)
    group_correct = torch.zeros(2, 2)
    
    with torch.no_grad():
        # Generate a balanced test set to see true performance
        x_test, y_test, a_test = create_waterbirds_batch(batch_size=num_samples, minority_ratio=0.5)
        _, robust_logits = model(x_test)
        preds = torch.argmax(robust_logits, dim=1)
        
        for i in range(num_samples):
            y, a = y_test[i].item(), a_test[i].item()
            group_counts[y, a] += 1
            if preds[i] == y:
                group_correct[y, a] += 1
                
    group_accs = group_correct / group_counts
    avg_acc = group_accs.mean().item()
    worst_acc = group_accs.min().item()
    
    return {
        "Group Accuracies": group_accs.numpy(),
        "Average Accuracy (GBA)": avg_acc,
        "Worst-Group Accuracy (WGA)": worst_acc
    }

# Compare Metrics
print("\n--- ERM Results ---")
print(evaluate_robustness(model_erm))

print("\n--- Logit Correction Results ---")
print(evaluate_robustness(model_lc))


--- ERM Results ---
{'Group Accuracies': array([[0.32780084, 0.22137405],
       [0.6091954 , 0.80508476]], dtype=float32), 'Average Accuracy (GBA)': 0.49086374044418335, 'Worst-Group Accuracy (WGA)': 0.221374049782753}

--- Logit Correction Results ---
{'Group Accuracies': array([[0.38247013, 0.802521  ],
       [0.9004525 , 0.56206894]], dtype=float32), 'Average Accuracy (GBA)': 0.6618781089782715, 'Worst-Group Accuracy (WGA)': 0.38247013092041016}


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PolynomialTailedLoss(nn.Module):
    def __init__(self, K=2):
        """
        K: The power/degree of the polynomial tail. 
        Higher K makes the loss steeper near the decision boundary.
        """
        super(PolynomialTailedLoss, self).__init__()
        self.K = K

    def forward(self, logits, labels):
        # Calculate the margin: y * f(x)
        # For binary classification with logits:
        # labels should be converted to {-1, 1}
        y = (2 * labels - 1).float()
        # We assume binary classification for Waterbirds (y_hat is the logit for class 1)
        # If multi-class, we use the margin of the correct class vs others.
        margins = y * logits[:, 1] 
        
        # Polynomial tail formula: 1 / (1 + margin^K)
        # We add a small constant to ensure the denominator is never zero
        # and handle negative margins by shifting/clipping as per standard implementations
        # to ensure it remains a valid loss (decreasing in margin).
        loss = 1.0 / (1.0 + torch.pow(torch.clamp(margins, min=0) + 1.0, self.K))
        
        # Alternatively, a more stable version for gradient descent:
        # loss = torch.mean(torch.log(1 + torch.pow(torch.abs(margins - target), self.K)))
        return loss.mean()

In [8]:
from torch.utils.data import Dataset, DataLoader

class WaterbirdsDataset(Dataset):
    def __init__(self, num_samples=1000, minority_ratio=0.5):
        # We use a 0.5 ratio for testing to ensure a balanced evaluation 
        # of all 4 groups (Landbird/Land, Landbird/Water, etc.) [cite: 302, 308]
        self.x, self.y, self.a = create_waterbirds_batch(num_samples, minority_ratio)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.a[idx]

test_dataset = WaterbirdsDataset(num_samples=1000, minority_ratio=0.5)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

def calculate_metrics(model, dataloader):
    """
    Calculates Average Accuracy, Group-Balanced Accuracy (GBA), 
    and Worst-Group Accuracy (WGA) as defined in the papers.
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_attrs = []
    
    with torch.no_grad():
        for x, y, a in dataloader:
            _, robust_logits = model(x)
            preds = torch.argmax(robust_logits, dim=1)
            all_preds.append(preds)
            all_labels.append(y)
            all_attrs.append(a)
            
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    attrs = torch.cat(all_attrs)
    
    # Define the 4 Waterbirds groups: (Landbird/Land, Landbird/Water, etc.)
    group_accuracies = []
    for y_val in [0, 1]:
        for a_val in [0, 1]:
            mask = (labels == y_val) & (attrs == a_val)
            if mask.sum() > 0:
                acc = (preds[mask] == labels[mask]).float().mean().item()
                group_accuracies.append(acc)
    
    metrics = {
        "Average Accuracy": (preds == labels).float().mean().item(),
        "Worst-Group Accuracy (WGA)": min(group_accuracies),
        "Group-Balanced Accuracy (GBA)": sum(group_accuracies) / len(group_accuracies)
    }
    return metrics

# Training logic for Polynomial Tailed Loss
def train_poly_model(K=2):
    model = TwoBranchModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    poly_criterion = PolynomialTailedLoss(K=K)
    
    for epoch in range(50):
        x, y, a = create_waterbirds_batch() # Minority ratio 0.05
        optimizer.zero_grad()
        
        # For the poly-tailed experiment, we focus on the robust branch directly
        _, robust_logits = model(x)
        loss = poly_criterion(robust_logits, y)
        
        loss.backward()
        optimizer.step()
    return model

# Final Comparison
print("--- Comparison of Methods ---")
erm_results = calculate_metrics(model_erm, test_loader)
lc_results = calculate_metrics(model_lc, test_loader)
poly_results = calculate_metrics(train_poly_model(K=2), test_loader)

for name, res in zip(["ERM (Cross-Entropy)", "Logit Correction", "Polynomial-Tailed"], 
                     [erm_results, lc_results, poly_results]):
    print(f"\n{name}:")
    for k, v in res.items():
        print(f"  {k}: {v:.4f}")

--- Comparison of Methods ---

ERM (Cross-Entropy):
  Average Accuracy: 0.5210
  Worst-Group Accuracy (WGA): 0.2550
  Group-Balanced Accuracy (GBA): 0.5221

Logit Correction:
  Average Accuracy: 0.6670
  Worst-Group Accuracy (WGA): 0.4818
  Group-Balanced Accuracy (GBA): 0.6613

Polynomial-Tailed:
  Average Accuracy: 0.4340
  Worst-Group Accuracy (WGA): 0.0000
  Group-Balanced Accuracy (GBA): 0.4495


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# =================================================================
# 1. PARAMETRIC POLYNOMIAL LOSS CLASS
# =================================================================
class ParametricPolyLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=1.0, c=1.0):
        """
        Experimental Polynomial Loss to test sensitivity to different components.
        - alpha: Decay rate of the tail. 
        - beta: The margin threshold where polynomial behavior begins. 
        - c: Scaling factor for importance weights (exponentiation). 
        """
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.c = c

    def forward(self, logits, labels, base_weights=None):
        # Calculate margins (z = y * f(x))
        # For binary classification, we use the difference between the two logit outputs.
        y_signed = (2 * labels - 1).float()
        z = y_signed * (logits[:, 1] - logits[:, 0])
        
        # 1. Polynomial Tail component: 1 / [z - (beta-1)]^alpha 
        # This part ensures importance weights are not ignored during interpolation. 
        poly_margin = z - (self.beta - 1)
        tail_loss = 1.0 / torch.pow(torch.clamp(poly_margin, min=0.1), self.alpha)
        
        # 2. Left/Logistic component for numerical stability at small margins. 
        # Normalized so that the loss is continuous at the switchover point z = beta. 
        left_loss = torch.log(1 + torch.exp(-z)) / 0.6931 
        
        # Combine components
        mask = (z >= self.beta).float()
        loss = (mask * tail_loss) + ((1 - mask) * left_loss)
        
        # 3. Apply Exponentiated Importance Weights 
        # Theory shows w^c (where c > 1) can significantly improve performance. 
        if base_weights is not None:
            effective_weights = torch.pow(base_weights, self.c)
            loss = loss * effective_weights
            
        return loss.mean()

# =================================================================
# 2. DATA AND MODEL SETUP
# =================================================================
def get_synthetic_data(n=2000, bias_ratio=0.95):
    # Simulates a dataset where features are biased towards a shortcut.
    y = torch.randint(0, 2, (n,))
    a = torch.where(torch.rand(n) < bias_ratio, y, 1 - y) # Spurious attribute
    
    x = torch.randn(n, 10)
    x[:, 0] = (2 * y.float() - 1) * 1.0  # Stable feature
    x[:, 1] = (2 * a.float() - 1) * 3.0  # Shortcut feature (Stronger)
    return x, y, a

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    def forward(self, x):
        return None, self.fc(x) # Return None for first branch to maintain compatibility

# =================================================================
# 3. ABLATION STUDY SCRIPT
# =================================================================
def run_ablation_study():
    # Test parameters: Varying alpha (decay) and c (weight exponent)
    alphas = [1.0, 2.0, 4.0]
    exponents = [1.0, 2.0, 3.0]
    
    test_x, test_y, test_a = get_synthetic_data(1000, 0.5) # Balanced test set
    
    print(f"{'Alpha':<10} | {'Exp (c)':<10} | {'Worst-Group Acc':<15} | {'Avg Acc':<15}")
    print("-" * 45)

    for alpha in alphas:
        for c in exponents:
            model = SimpleModel()
            optimizer = optim.Adam(model.parameters(), lr=1e-3)
            criterion = ParametricPolyLoss(alpha=alpha, beta=1.0, c=c)
            
            # Training
            for epoch in range(20):
                x, y, a = get_synthetic_data(1000, 0.95)
                weights = torch.ones_like(y).float()
                weights[y != a] = 10.0 # Standard importance weight for minority groups 
                
                optimizer.zero_grad()
                _, logits = model(x)
                loss = criterion(logits, y, base_weights=weights)
                loss.backward()
                optimizer.step()
            
            # Evaluation
            model.eval()
            with torch.no_grad():
                _, out = model(test_x)
                preds = torch.argmax(out, dim=1)
                
                group_accs = []
                for y_v in [0, 1]:
                    for a_v in [0, 1]:
                        mask = (test_y == y_v) & (test_a == a_v)
                        acc = (preds[mask] == test_y[mask]).float().mean().item()
                        group_accs.append(acc)
                
                wga = min(group_accs)
                avg_acc = sum(group_accs) / len(group_accs)
                print(f"{alpha:<10.1f} | {c:<10.1f} | {wga:<15.4f} | {avg_acc:<15.4f}")

if __name__ == "__main__":
    run_ablation_study()

Alpha      | Exp (c)    | Worst-Group Acc | Avg Acc        
---------------------------------------------
1.0        | 1.0        | 0.0082          | 0.5037         
1.0        | 2.0        | 0.3074          | 0.6253         
1.0        | 3.0        | 0.1633          | 0.4174         
2.0        | 1.0        | 0.3938          | 0.6819         
2.0        | 2.0        | 0.3130          | 0.6922         
2.0        | 3.0        | 0.0270          | 0.5017         
4.0        | 1.0        | 0.4344          | 0.4995         
4.0        | 2.0        | 0.0574          | 0.3862         
4.0        | 3.0        | 0.0123          | 0.4475         
